# Four independent disease classifiers

Mentor notebook for MediAI. Each disease is its **own** binary model. There is no combined four-disease network.

**What this notebook actually does (not a sketch):** every training cell below **fits** XGBoost, LightGBM, CatBoost and TabPFN. Optuna trial lines are printed as they run (CV ROC-AUC, F1, accuracy, hyperparameters). The winning algorithm is refit, scored on a frozen holdout, explained with SHAP, and saved.

TabPFN uses the **open TabPFNv2** checkpoint (Prior-Labs/TabPFN-v2-clf). TabPFN 9.x would otherwise default to gated v3.5 weights that need a Prior Labs login; that is not used here.

These are screening-style classifiers on public tables. They are **not** diagnoses and not clinician-signed.

| Disease | Training rows | Why this data |
|---|---|---|
| Heart | Indian hospital (n=1000) **plus** UCI Cleveland / Hungary / Switzerland / VA | India-first, same 13 clinical fields as the app; extra sites for pooling |
| Liver | ILPD Andhra Pradesh **plus** UCI HCV labs | ILPD matches the app form; HCV appended with ILPD-only labs left missing |
| Diabetes | Pima 8-lab table | Only public table that matches the app form. Zeros in glucose/BP/skin/insulin/BMI were recoded to missing. Not an Indian cohort. |
| Kidney | UCI CKD, Karaikudi, Tamil Nadu | India. Missing labs were class-blind imputed so “lab not ordered” cannot leak the label. Bangladesh bins used only as secondary. |

**Protocol (identical for every disease):** stratified 80/20 holdout → Optuna on the 80% maximizing 5-fold (3-fold for TabPFN) **ROC-AUC** → pick winner by CV AUC, then F1, then recall → score the 20% once → SHAP on the winner.


In [ ]:
import json, os, sys, time
from pathlib import Path
from IPython.display import display, Image, Markdown
import pandas as pd
import matplotlib
matplotlib.use("Agg")

os.environ.setdefault("TABPFN_MODEL_VERSION", "v2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "2")

ROOT = Path.cwd()
if not (ROOT / "ml" / "train_compare.py").exists():
    # notebook executed from notebooks/
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
print("project root:", ROOT)
print("python:", sys.version.split()[0])

from ml.data_prep import (
    feature_target, load_heart, load_liver, load_diabetes, load_kidney, load_diabetes_sylhet,
)
from ml.run_training import train_heart, train_liver, train_diabetes, train_kidney
from ml.train_compare import TREE_TRIALS, TABPFN_TRIALS, TREE_FOLDS, TABPFN_FOLDS

print(f"Optuna budget: trees {TREE_TRIALS} trials × {TREE_FOLDS}-fold; TabPFN {TABPFN_TRIALS} × {TABPFN_FOLDS}-fold")
print("TabPFN checkpoint: open v2 (TABPFN_MODEL_VERSION=v2)")
RESULTS = {}


---
## 1. Heart disease

**Task.** Presence vs absence of heart disease (binary). UCI `num>0`; Indian `target`.

**Why these datasets.** The app already collects Cleveland-style fields (age, sex, chest pain, BP, cholesterol, ECG, max HR, ST depression, vessels). A 1000-row Indian hospital table (Mendeley DOI 10.17632/dzz48mvjht.1) uses the same panel (no `thal`). UCI Cleveland / Hungary / Switzerland / VA add extra sites with that schema. Chest-pain codes on the India table are 0–3 and are shifted to UCI’s 1–4 before pooling. The India file is not a Cleveland clone (0 overlapping rows on age+BP+cholesterol+max HR).

**Not a diagnosis.** Angiographic tables from the 1980s plus one Indian hospital dump.


In [ ]:
heart_pool, heart_india = load_heart()
print("pooled rows", len(heart_pool), "by source:\n", heart_pool["source"].value_counts().to_string())
print("class balance (1=disease):")
print(heart_pool.groupby("source")["disease"].mean().round(3).to_string())
display(heart_pool.drop(columns=["source"]).head(8))
print("missing counts:\n", heart_pool.isna().sum().to_string())


The next cell **trains** all four algorithms. Each printed `trial` line is a real cross-validated fit, not a placeholder.

In [ ]:
t0 = time.time()
RESULTS["heart"] = train_heart()
print(f"\nheart wall time: {time.time()-t0:.0f}s")
print("winner:", RESULTS["heart"]["winner"])
display(pd.DataFrame(RESULTS["heart"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["heart"]
print("Why this winner:", h["winner"],
      "had the highest cross-validated ROC-AUC among the four tuned algorithms.")
print("CV:", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout:", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Holdout by hospital source:", json.dumps(h.get("winner_holdout_by_source"), indent=2, default=str)[:1500])
print("Best hyperparameters:", h["winner_params"])
print("\n--- Optuna trial logs (every algorithm that fitted) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("\nSHAP: bar = mean |impact|; beeswarm = direction per patient on the train split.")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        print(path)
        display(Image(filename=str(path)))


---
## 2. Liver disease

**Task.** Liver-disease vs not. ILPD `Selector==1`; HCV non-donor vs blood donor (suspect donors dropped).

**Why these datasets.** ILPD (DOI 10.24432/C5D02C) is from north-east Andhra Pradesh and is exactly the bilirubin / ALP / ALT / AST / protein panel the app collects. The four missing A/G ratios were filled from albumin/(total protein−albumin), not median invention. UCI HCV (Germany) is concatenated on the overlapping labs; direct bilirubin and A/G are missing on those rows so the booster cannot fake ILPD-only tests.


In [ ]:
liver_pool, ilpd = load_liver()
print("pooled rows", len(liver_pool))
print(liver_pool.groupby("source")["disease"].agg(["size","mean"]))
display(ilpd.head(6))
print("ILPD A/G missing after cleaning:", int(ilpd["AG"].isna().sum()) if "AG" in ilpd.columns else int(ilpd.filter(like="AG").isna().sum().sum()))


In [ ]:
t0 = time.time()
RESULTS["liver"] = train_liver()
print(f"\nliver wall time: {time.time()-t0:.0f}s")
display(pd.DataFrame(RESULTS["liver"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["liver"]
print("Winner", h["winner"], "selected by CV ROC-AUC.")
print("CV", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Holdout by source (ILPD vs HCV):", json.dumps(h.get("winner_holdout_by_source"), indent=2, default=str)[:1500])
print("Params", h["winner_params"])
print("\n--- Optuna trial logs (every algorithm that fitted) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("\nNote: ILPD-only holdout AUC is expected to be lower than pooled AUC. HCV donors vs hepatitis is an easier split than ILPD's mixed liver panel.")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        display(Image(filename=str(path)))


---
## 3. Diabetes

**Task.** WHO-style diabetes present vs absent on the 8-lab form.

**Why this dataset.** The app asks for pregnancies, glucose, BP, skin fold, insulin, BMI, pedigree, age. That is the Pima table. It is **not** Indian (Gila River / Akimel O’odham women ≥21). Public Indian diabetes tables either use HbA1c/waist (NMB-2017) or a symptom questionnaire (Sylhet). Those cannot be concatenated onto this form.

**Cleaning that was required.** Zeros in glucose, BP, skin, insulin, BMI are structurally missing, not true zeros. They are NA here. `pregnancies=0` is kept.


In [ ]:
dia = load_diabetes()
print(dia["disease"].value_counts().to_string())
print("NA after zero-fix:\n", dia.isna().sum().to_string())
display(dia.head(8))
syl = load_diabetes_sylhet()
print("\nSylhet early-stage (NOT pooled — different schema):", syl.shape, "positive rate", float(syl.disease.mean()))
print("Sylhet columns:", list(syl.columns))


In [ ]:
t0 = time.time()
RESULTS["diabetes"] = train_diabetes()
print(f"\ndiabetes wall time: {time.time()-t0:.0f}s")
display(pd.DataFrame(RESULTS["diabetes"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["diabetes"]
print("Winner", h["winner"], "— highest CV ROC-AUC on the Pima 8-lab table.")
print("CV", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Params", h["winner_params"])
print("\n--- Optuna trial logs (every algorithm that fitted) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("SHAP should highlight glucose, BMI, age, insulin if the model is using the labs the form collects.")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        display(Image(filename=str(path)))


---
## 4. Chronic kidney disease

**Task.** `ckd` vs `notckd`.

**Why this dataset.** UCI CKD (DOI 10.24432/C5G020) is from Apollo, Karaikudi, Tamil Nadu — the India table. Two dirty labels (`ckd\t`) were stripped. Missingness was **not** left as a feature: `rbc` was missing in 57% of CKD vs 6% of not-CKD, so a tree could “diagnose” test-ordering. Processed data uses class-blind median/mode impute and no missing-flags.

Bangladesh UCI 857 is a second hospital but labs are **binned** (and some Excel date artifacts). It is scored as secondary on shared columns except `bp` (0/1 there vs mmHg here). `stage` / `grf` / `affected` are never used (label leakage).


In [ ]:
kid_p, kid_s = load_kidney()
print("Tamil Nadu", kid_p.shape, kid_p["disease"].value_counts().to_dict())
print("Bangladesh secondary", kid_s.shape, kid_s["disease"].value_counts().to_dict())
display(kid_p.head(6))
print("primary missing cells (must be 0):", int(kid_p.drop(columns=["source"]).isna().sum().sum()))


In [ ]:
t0 = time.time()
RESULTS["kidney"] = train_kidney()
print(f"\nkidney wall time: {time.time()-t0:.0f}s")
display(pd.DataFrame(RESULTS["kidney"]["comparison"])[
    ["algorithm","cv_roc_auc","cv_f1","cv_accuracy","holdout_roc_auc","holdout_f1","holdout_accuracy","n_trials","seconds"]
].round(4))


In [ ]:
h = RESULTS["kidney"]
print("Winner", h["winner"], "on Tamil Nadu CKD after leak-blocked impute.")
print("CV", {k: round(v,4) for k,v in h["winner_cv"].items() if k!="folds"})
print("Holdout", {k: round(v,4) for k,v in h["winner_holdout"].items()})
print("Secondary Bangladesh:", h.get("secondary"))
print("Params", h["winner_params"])
print("\n--- Optuna trial logs (every algorithm that fitted) ---")
for algo, lines in h["trial_logs"].items():
    print(f"\n[{algo}] {len(lines)} trials")
    for line in lines:
        print(line)
print("UCI CKD is unusually separable on hemoglobin / packed-cell volume / hypertension even after class-blind impute. Near-1.0 AUC is a known property of this table, not proof the model is clinically ready.")
print("If SHAP is dominated by a single lab that was heavily missing in the raw file, treat that with caution even after impute.")
for p in h.get("shap") or []:
    path = Path(p)
    if path.exists():
        display(Image(filename=str(path)))


---
## 5. Summary (mentor table)

Holdout metrics for the **winning algorithm per disease**. Accuracy is reported; it was not used to pick the winner (ROC-AUC was).


In [ ]:
rows = []
ds_used = {
    "heart": "India hospital + UCI 4 sites (Cleveland/Hungary/Switzerland/VA)",
    "liver": "ILPD Andhra Pradesh + UCI HCV",
    "diabetes": "Pima 8-lab (zeros→NA); Sylhet not pooled",
    "kidney": "UCI CKD Tamil Nadu; Bangladesh secondary",
}
for disease, h in RESULTS.items():
    ho = h["winner_holdout"]
    rows.append({
        "disease": disease,
        "winning_algorithm": h["winner"],
        "accuracy": round(ho["accuracy"], 4),
        "F1": round(ho["f1"], 4),
        "ROC-AUC": round(ho["roc_auc"], 4),
        "dataset_used": ds_used[disease],
    })
summary = pd.DataFrame(rows)
display(summary)
(ROOT / "ml" / "artifacts" / "summary.json").write_text(json.dumps(rows, indent=2))
print("models:")
for disease, h in RESULTS.items():
    print(" ", h["model_path"])


### How to read the SHAP plots

- **Bar:** mean |SHAP| — which fields moved the score most on the train split.
- **Beeswarm:** each dot is a patient. Color is the field value. Right of zero pushed risk **up**.

If two algorithms’ CV-AUC differ by less than ~0.01, the pick is a coin flip for this sample size. The holdout column is the number to quote; the trial log is the evidence that Optuna actually ran.

Not for clinical use.
